In [1]:
import pandas as pd
import numpy as np
import os
import glob

import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
import requests
current_date = "2025-12"

my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"

end_use = "naics?get=ALL_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL"

url = "https://api.census.gov/data/timeseries/intltrade/exports/" + end_use 
url = url + my_key + "&time==from+2013-01"

r = requests.get(url)

df = pd.DataFrame(r.json()[1:])
df.columns = r.json()[0]
df["total_exports"] = df["ALL_VAL_MO"].astype(float)
df = df[df.SUMMARY_LVL == "DET"]

grp = df.groupby(["CTY_NAME"])
top_products = grp.agg({"total_exports": "sum", "CTY_CODE": "first"})

country_list = list(top_products.sort_values(by="total_exports", ascending=False).CTY_CODE)[0:31]
country_list[0] = ""
country_list.extend(["0003", "0020"])

In [ ]:
# data_dir = ".\\data\\exports-hs10"
# only run if in the folder locally

# for xxx in country_list:
    
#     country_label = "TOTAL" if xxx == "" else xxx
    
#     # Find all monthly files for this country
#     pattern = os.path.join(data_dir, f"{country_label}data-*.parquet")
#     monthly_files = sorted(glob.glob(pattern))
    
#     # Exclude any existing combined file from the list
#     combined_file = os.path.join(data_dir, f"{country_label}data-combined.parquet")
#     monthly_files = [f for f in monthly_files if "combined" not in f]
    
#     if not monthly_files:
#         print(f"No files found for {country_label}, skipping...")
#         continue
    
#     print(f"Combining {len(monthly_files)} monthly files for {country_label}...")
    
#     dfs = []
#     for f in monthly_files:
#         try:
#             dfs.append(pq.read_table(f).to_pandas())
#         except Exception as e:
#             print(f"  Error reading {f}: {e}")
    
#     if not dfs:
#         print(f"  No data loaded for {country_label}, skipping...")
#         continue
    
#     combined = pd.concat(dfs, ignore_index=True)
    
#     pq.write_table(pa.Table.from_pandas(combined), combined_file)
    
#     print(f"  Saved {len(combined):,} rows to {combined_file}")

# print("Done!")

In [3]:
data_dir = ".\\data\\exports-hs10"

# Find all country-level combined files, excluding TOTAL and USMCA (0020)
exclude_labels = {"TOTAL", "0020"}

all_combined = sorted(glob.glob(os.path.join(data_dir, "*data-combined.parquet")))
country_files = [
    f for f in all_combined
    if not any(os.path.basename(f).startswith(lbl + "data-") for lbl in exclude_labels)
]

print(f"Combining {len(country_files)} country files into All-country-exports.parquet...")

dfs = []
for f in country_files:
    try:
        dfs.append(pq.read_table(f).to_pandas())
    except Exception as e:
        print(f"  Error reading {f}: {e}")

if dfs:
    all_countries = pd.concat(dfs, ignore_index=True)
    out_path = os.path.join(data_dir, "All-country-exports.parquet")
    pq.write_table(pa.Table.from_pandas(all_countries), out_path)
    print(f"Saved {len(all_countries):,} rows to {out_path}")
else:
    print("No data loaded, nothing saved.")

Combining 31 country files into All-country-exports.parquet...
Saved 22,684,566 rows to .\data\exports-hs10\All-country-exports.parquet
